In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import t

# === 全局样式（沿用代码 A） ===
mpl.rc('xtick', direction='out', labelsize=8)
mpl.rc('ytick', direction='out', labelsize=8)
mpl.rcParams['xtick.major.width'] = 0.8
mpl.rcParams['ytick.major.width'] = 0.8
mpl.rc('font', family='sans-serif')
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Liberation Sans']
mpl.rc('axes', titlesize=12, labelsize=10, linewidth=0.8)
mpl.rc('legend', fontsize=8, frameon=False)
mpl.rc('figure', figsize=(6,4), dpi=300)
mpl.rc('savefig', bbox='tight', pad_inches=0.1)

# === 路径与参数 ===
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
OUTPUT_DIR  = '/home/weiji/restart_exam/20250506/Vertical_Profiles_new/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
MONTHS      = range(1,6)
BASE_YRS    = [y for y in range(1,201) if y != 8]
REF_YEAR    = 8

# === 数据处理 ===
def process_variable(ds, var='O3'):
    da = ds[var].mean(dim='lon')
    if 'member' in da.dims:
        da = da.mean(dim='member')
    if var == 'O3':
        da = da.sel(lat=slice(60,90))
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean(dim='lat')
    else:
        raise ValueError(f"Unsupported var: {var}")
    return da


def bootstrap_ci(data, nboot=500, alpha=0.05):
    means = np.empty(nboot)
    n = data.size
    for i in range(nboot):
        samp = np.random.choice(data, size=n, replace=True)
        means[i] = samp.mean()
    return np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])

# === 读取气候态（日分辨率） ===
def read_o3_climatology():
    ds = xr.open_dataset(MERGED_FILE)
    da_all = process_variable(ds, 'O3')
    ds.close()
    base = da_all.sel(
        time=(da_all.time.dt.year.isin(BASE_YRS)) &
             (da_all.time.dt.month.isin(MONTHS))
    )
    clim = base.groupby('time.dayofyear').mean(dim='time')
    return clim  # dims: (dayofyear)

# === 读取参考年剖面 ===
def read_o3_reference():
    ds = xr.open_dataset(MERGED_FILE)
    da = process_variable(ds, 'O3')
    ds.close()
    ref = da.sel(
        time=(da.time.dt.year == REF_YEAR) &
             (da.time.dt.month.isin(MONTHS))
    )
    return ref  # dims: (time)

# === 计算异常并对齐 ===
def compute_anomaly(ref, clim):
    times_ref = ref.time.values
    doy_ref   = ref.time.dt.dayofyear.values
    clim_sel  = clim.sel(dayofyear=xr.DataArray(doy_ref, dims='time'))
    anom = xr.DataArray(
        ref.values.T - clim_sel.values.T,
        coords={'plev': ref.plev.values, 'time': times_ref},
        dims=['plev','time']
    )
    return anom, clim_sel

# === 统计显著性检验 ===
def compute_significance(clim, ref_anom):
    ds = xr.open_dataset(MERGED_FILE)
    da_all = process_variable(ds, 'O3')
    ds.close()
    base = da_all.sel(
        time=(da_all.time.dt.year.isin(BASE_YRS)) &
             (da_all.time.dt.month.isin(MONTHS))
    )
    doy_b  = base.time.dt.dayofyear.values
    clim_b = clim.sel(dayofyear=xr.DataArray(doy_b, dims='time'))
    base_anom = base.values.T - clim_b.values.T
    lev_n, time_n = ref_anom.shape
    t_mask = np.zeros((lev_n, time_n), bool)
    b_mask = np.zeros((lev_n, time_n), bool)
    for ti, day in enumerate(ref_anom.time.dt.dayofyear.values):
        samp = base_anom[:, doy_b == day]
        for li in range(lev_n):
            col = samp[li, :]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue
            obs = ref_anom.values[li, ti]
            se  = np.std(col, ddof=1) / np.sqrt(col.size)
            tstat = obs / se
            pval  = 2 * (1 - t.cdf(abs(tstat), df=col.size-1))
            t_mask[li, ti] = (pval < 0.05)
            lo, hi = bootstrap_ci(col)
            b_mask[li, ti] = not (lo <= obs <= hi)
    return t_mask, b_mask

# === 绘图函数（自代码 A） ===
def plot_merra2_o3_anomaly(anom_da, clim_da_sel, save_path,
                           unit='ppm', sig_mask=None,
                           show_nonsig=False):
    fig, ax = plt.subplots()
    tnum = mdates.date2num(anom_da.time.values)
    p_pa = anom_da.plev.values 
    levels = np.linspace(-1.5, 1.5, 31) if unit=='ppm' else 20
    cf = ax.contourf(
        tnum, p_pa, anom_da.values,
        levels=levels, cmap='RdBu_r',
        extend='both', antialiased=True, alpha=0.85
    )
    clim_vals = clim_da_sel.values.T
    if unit=='ppm':
        clim_lvls = [4.8,5.2,5.6,6.0,6.4]
        fmt = '%.1f'
    else:
        clim_lvls = [4.8e-6,5.2e-6,5.6e-6,6.0e-6,6.4e-6]
        fmt = '%.1e'
    CS = ax.contour(
        tnum, p_pa, clim_vals,
        levels=clim_lvls, colors='k', linewidths=1.0
    )
    ax.clabel(CS, inline=True, fontsize=8, fmt=fmt)
    if sig_mask is not None:
        mask = (~sig_mask) if show_nonsig else sig_mask
        legend_txt = 'Not significant (p ≥ 0.05)' if show_nonsig else 'Significant (p < 0.05)'
        sig_da = xr.DataArray(mask.astype(int), coords=anom_da.coords, dims=anom_da.dims)
        ax.contourf(
            tnum, p_pa, sig_da,
            levels=[0.5,1.5], colors='none',
            hatches=['///'], alpha=0
        )
        from matplotlib.patches import Patch
        patch = Patch(facecolor='white', hatch='///', edgecolor='black', label=legend_txt)
        ax.legend(handles=[patch], loc='upper right')
    ax.set_yscale('log')
    ax.invert_yaxis()
    ax.set_xlim(tnum[0], tnum[-1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.grid(True, which='major', linestyle='--', linewidth=0.4, alpha=0.5)
    ax.set_ylabel('Pressure (Pa)')
    ax.set_title(f'O$_3$ Anomaly (Year {REF_YEAR:04d})')
    cax = fig.add_axes([0.92,0.15,0.02,0.7])
    fig.colorbar(cf, cax=cax, label=f'O$_3$ Anomaly ({unit})')
    plt.savefig(save_path, dpi=150)
    plt.show(fig)

# === 主流程 ===
if __name__ == '__main__':
    clim = read_o3_climatology()
    ref  = read_o3_reference()
    anom, clim_sel = compute_anomaly(ref, clim)
    anom_ppm = anom * 1e6
    clim_ppm = clim_sel * 1e6
    t_mask, b_mask = compute_significance(clim, anom)

    plot_merra2_o3_anomaly(
        anom_ppm, clim_ppm,
        os.path.join(OUTPUT_DIR, 'anom_0008_ppm.png'),
        unit='ppm'
    )
    plot_merra2_o3_anomaly(
        anom_ppm, clim_ppm,
        os.path.join(OUTPUT_DIR, 't_0008_ppm.png'),
        unit='ppm', sig_mask=t_mask
    )
    plot_merra2_o3_anomaly(
        anom_ppm, clim_ppm,
        os.path.join(OUTPUT_DIR, 'b_0008_ppm.png'),
        unit='ppm', sig_mask=b_mask
    )
    plot_merra2_o3_anomaly(
        anom_ppm, clim_ppm,
        os.path.join(OUTPUT_DIR, 'ns_t_0008_ppm.png'),
        unit='ppm', sig_mask=t_mask, show_nonsig=True
    )
    plot_merra2_o3_anomaly(
        anom_ppm, clim_ppm,
        os.path.join(OUTPUT_DIR, 'ns_b_0008_ppm.png'),
        unit='ppm', sig_mask=b_mask, show_nonsig=True
    )

    print("Finished generating 0008 O3 vertical profile and significance plots.")